# AlphaZero Connect Four — Colab Training

Run cells top to bottom. Designed to survive Colab disconnects:
checkpoints are written to Google Drive, not the ephemeral Colab disk,
so a dropped session just means re-running this notebook and resuming.

**Before running:** Runtime → Change runtime type → T4 GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — stop and fix runtime type before continuing')

In [ ]:
# Mount Drive so checkpoints survive session disconnects
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_ROOT = '/content/drive/MyDrive/alphazero_connect4_checkpoints'
import os
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)

In [ ]:
# Clone the repo (replace URL once pushed to GitHub)
REPO_URL = 'https://github.com/PrakrutiBhaskar/alphazero-connect4.git'

!rm -rf alphazero-connect4
!git clone {REPO_URL}
%cd alphazero-connect4

In [ ]:
!pip install -q -r requirements.txt

## Sanity check before spending GPU time

Always run the smoke test first — it's seconds on CPU and catches
wiring bugs before they burn a training run.

In [ ]:
!python scripts/smoke_test.py

## Baseline training run

This points `checkpoint_dir` at Drive so progress survives a disconnect.
`run_training.py` currently starts fresh each call — see the note it
prints if a checkpoint already exists; wire up checkpoint loading in
`src/train.py` before relying on this for a true multi-session resume.

In [ ]:
CONFIG_NAME = 'baseline'
CKPT_DIR = f'{CHECKPOINT_ROOT}/{CONFIG_NAME}'

!python scripts/run_training.py --config {CONFIG_NAME} --checkpoint-dir {CKPT_DIR}

## Plot training curves

Run after (or during, on a completed prefix of) a training run to
visualize policy/value loss and win-rate-vs-previous-best over iterations.

In [ ]:
import json
import matplotlib.pyplot as plt

with open(f'{CKPT_DIR}/history.json') as f:
    history = json.load(f)

iters = [h['iteration'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(iters, [h['policy_loss'] for h in history], label='policy loss')
axes[0].plot(iters, [h['value_loss'] for h in history], label='value loss')
axes[0].set_xlabel('iteration'); axes[0].set_title('Training loss'); axes[0].legend()

axes[1].plot(iters, [h['win_rate_vs_prev_best'] for h in history])
axes[1].axhline(0.55, color='gray', linestyle='--', label='promotion threshold')
axes[1].set_xlabel('iteration'); axes[1].set_title('Win rate vs previous best'); axes[1].legend()

axes[2].plot(iters, [h['replay_buffer_size'] for h in history])
axes[2].set_xlabel('iteration'); axes[2].set_title('Replay buffer size')

plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/training_curves.png', dpi=150)
plt.show()

## Ablation runs

Run each named config separately (each is its own checkpoint dir under
Drive, so they don't clobber each other or the baseline). Do these
*after* confirming the baseline trains cleanly — no point ablating a
broken loop.

In [ ]:
ABLATIONS = ['sims_50', 'sims_800', 'no_dirichlet_noise', 'shallow_net', 'wide_net']

# Run one at a time (uncomment as you go — don't queue all 5 blind,
# check each finishes cleanly before starting the next):

# CONFIG_NAME = 'sims_50'
# !python scripts/run_training.py --config {CONFIG_NAME} --checkpoint-dir {CHECKPOINT_ROOT}/{CONFIG_NAME}